# Day 01 — Solution Notebook
Introduction to Agentic AI: minimal agent loop + exercises walkthrough.

## Setup

In [ ]:
import os, json
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

## 1. The calculator tool

In [ ]:
def calculator(expression: str) -> str:
    allowed_chars = set('0123456789+-*/(). ')
    if not set(expression) <= allowed_chars:
        return 'Error: invalid characters in expression'
    try:
        return str(eval(expression, {'__builtins__': {}}))
    except ZeroDivisionError:
        return 'Error: division by zero'
    except Exception as e:
        return f'Error: {e}'

assert calculator('2 + 2') == '4'
assert calculator('1 / 0') == 'Error: division by zero'
print('calculator tool OK')

## 2. Tool schema + agent loop

In [ ]:
TOOLS = [{
    'type': 'function',
    'function': {
        'name': 'calculator',
        'description': "Evaluate a basic arithmetic expression, e.g. '12 * (3 + 4)'",
        'parameters': {
            'type': 'object',
            'properties': {'expression': {'type': 'string', 'description': 'A valid arithmetic expression'}},
            'required': ['expression'],
        },
    },
}]
AVAILABLE_FUNCTIONS = {'calculator': calculator}

def run_agent(user_goal: str, max_steps: int = 5) -> str:
    messages = [
        {'role': 'system', 'content': 'You are a careful assistant. Use the calculator tool for any arithmetic instead of computing it yourself.'},
        {'role': 'user', 'content': user_goal},
    ]
    for step in range(max_steps):
        response = client.chat.completions.create(model='gpt-4o-mini', messages=messages, tools=TOOLS)
        msg = response.choices[0].message
        if msg.tool_calls:
            messages.append(msg)
            for tool_call in msg.tool_calls:
                fn_name = tool_call.function.name
                fn_args = json.loads(tool_call.function.arguments)
                print(f'[step {step}] Agent calls {fn_name}({fn_args})')
                result = AVAILABLE_FUNCTIONS[fn_name](**fn_args)
                messages.append({'role': 'tool', 'tool_call_id': tool_call.id, 'content': result})
            continue
        return msg.content
    return 'Max steps reached without a final answer.'

## 3. Run it (requires OPENAI_API_KEY in your .env)

In [ ]:
# Uncomment to run live against the API:
# answer = run_agent('What is (482 * 17) - 96, and then explain briefly why agents need tools for math?')
# print(answer)

## 4. Exercise 5 solution — division-by-zero guard
Already incorporated above via the `ZeroDivisionError` branch.

## 5. Exercise 6 solution — word_count tool

In [ ]:
def word_count(text: str) -> int:
    return len(text.split())

assert word_count('the quick brown fox') == 4
print('word_count tool OK')

## Summary
You built and tested a minimal, framework-free agent loop, added a safety guard, and extended it with a second tool — exactly the skills Day 2 builds on.